In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Hallucination_RAG_Project')

Mounted at /content/drive


In [2]:
# 1. Install required vector and embedding libraries
!pip install faiss-cpu sentence-transformers groq -q

import os
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq
from getpass import getpass

# 2. Securely connect to Groq (if you didn't save the key in Colab's secrets)
if "GROQ_API_KEY" not in os.environ:
    print("Please enter your Groq API Key:")
    os.environ["GROQ_API_KEY"] = getpass()
client = Groq()

# 3. Load the dataset we prepared in Step 2
print("\nLoading dataset...")
df = pd.read_csv("./data/prototype_qa_data.csv")
knowledge_base = df['knowledge'].dropna().tolist()

# 4. STEP 5: Build Embeddings (Using the base paper's exact model)
print("Loading all-MiniLM-L6-v2 embedding model...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print(f"Embedding {len(knowledge_base)} knowledge documents. This will take a few seconds...")
# Normalizing vectors allows us to use Inner Product for cosine similarity
document_embeddings = embedder.encode(knowledge_base, convert_to_numpy=True, normalize_embeddings=True)

# 5. STEP 6: Build FAISS Vector Database
print("Building FAISS index...")
dimension = document_embeddings.shape[1] # 384 dimensions for MiniLM
index = faiss.IndexFlatIP(dimension)     # IP = Inner Product (Cosine Similarity)
index.add(document_embeddings)
print(f"FAISS index built successfully with {index.ntotal} vectors.")

# 6. Build the Retrieval Function
def retrieve_top_k(query, k=1):
    """Embeds the query and fetches the top K most relevant knowledge chunks."""
    query_embedding = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = index.search(query_embedding, k)

    retrieved_chunks = []
    for idx in indices[0]:
        if idx != -1:
            retrieved_chunks.append(knowledge_base[idx])
    return retrieved_chunks

# 7. STEP 7: Build SYSTEM B (Naive RAG)
def system_b_naive_rag(question, k=1):
    """
    Retrieves Top-K context and forces Llama to answer based ONLY on the context.
    """
    retrieved_contexts = retrieve_top_k(question, k)
    context_block = "\n\n".join(retrieved_contexts)

    prompt = f"""You are a helpful AI assistant. Answer the user's question based strictly on the provided context. If the context does not contain the answer, say "I don't know based on the provided context."

CONTEXT:
{context_block}

QUESTION:
{question}
"""
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=200
        )
        return response.choices[0].message.content.strip(), retrieved_contexts
    except Exception as e:
        return f"Error: {str(e)}", []

# 8. Test System B on the exact same question System A failed on
test_question = "Which magazine was started first Arthur's Magazine or First for Women?"
print(f"\nQUESTION: {test_question}")

system_b_answer, retrieved_docs = system_b_naive_rag(test_question, k=1)

print("\n--- SYSTEM B (Naive RAG) OUTPUT ---")
print(f"RETRIEVED CONTEXT:\n{retrieved_docs[0]}")
print(f"\nANSWER:\n{system_b_answer}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.8 MB/s eta 0:00:00
Please enter your Groq API Key:
··········

Loading dataset...
Loading all-MiniLM-L6-v2 embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 500 knowledge documents. This will take a few seconds...
Building FAISS index...
FAISS index built successfully with 500 vectors.

QUESTION: Which magazine was started first Arthur's Magazine or First for Women?

--- SYSTEM B (Naive RAG) OUTPUT ---
RETRIEVED CONTEXT:
Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.

ANSWER:
Based on the provided context, Arthur's Magazine was published from 1844-1846, while the information about First for Women mentions it is published by Bauer Media Group in the USA, but the start year is not provided. However, since First for Women is mentioned as being published by Bauer Media Group in the USA, it is likely that it was published after Arthur's Magazine, which was specifically stated to be published in the 19th century. 

However, since the exact start year of First for Women is not provided, we can